# Module 2: Shipment history in Iceberg

Contoso Fresh writes five years of shipment history as Iceberg tables in OneLake, applies a late carrier file, and traces the lots related to YG-2291 with Spark, DuckDB, AI Functions, and a Data Agent. Run the cells in order.

## Exercise 2.1: Prepare the notebook and data

### Cell 1: Generate data

Writes the Contoso Fresh CSV files to **Files/reference** and **Files/history**.

In [ ]:
# Contoso Fresh data generator
#
# Writes the current-state tables and five years of shipment history as CSV files.
# The seed is fixed, so every run writes the same files.
# In Fabric, the files go to Files/reference and Files/history in the default lakehouse.
# Anywhere else, the cell writes only the current-state files, to data/reference in the repo.

import bisect
import csv
import importlib.util
import math
import random
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

SEED = 2291
PARIS = ZoneInfo("Europe/Paris")
UTC = timezone.utc
AS_OF = datetime(2026, 10, 13, 6, 0, tzinfo=PARIS).astimezone(UTC)  # scenario clock, used instead of now()
FIRST_DAY = date(2021, 10, 13)
LAST_DELIVERY_DAY = date(2026, 10, 12)
RECALL_DAY = date(2026, 10, 11)
AS_OF_DAY = date(2026, 10, 13)

# Realism. Row counts follow from these numbers.
LINES_PER_SHIPMENT = 10
STORES_AT_START = 60
LATE_SHIPMENT_RATE = 0.025
SHORT_LINE_RATE = 0.012
REFUSED_LINE_RATE = 0.004
ISSUE_NOTE_RATE = 0.5
ROUTINE_NOTE_RATE = 0.03
FRENCH_NOTE_SHARE = 0.33
QA_NOTE_RATE = 0.05
COST_GROWTH_PER_YEAR = 0.03
CLOSED_DAYS = {(1, 1), (5, 1), (12, 25)}

MON, TUE, WED, THU, FRI, SAT, SUN = range(7)
MON_TO_SAT = (MON, TUE, WED, THU, FRI, SAT)

SUPPLIERS = [
    ("SUP-008", "Fabrikam Produce Partners", "Fresh produce", "France"),
    ("SUP-014", "Alpine Valley Dairy Cooperative", "Cultured dairy", "France"),
    ("SUP-021", "Northwind Frozen Foods", "Frozen prepared foods", "France"),
    ("SUP-027", "Woodgrove Creamery", "Cheese and dairy ingredients", "France"),
]

PRODUCT_FIELDS = ("id", "code", "name", "supplier", "category", "pack_size", "shelf_life", "max_c",
                  "units_per_case", "unit_cost", "demand", "season", "runs")
PRODUCTS = [
    # id, lot code, name, supplier, category, pack size, shelf life days, max storage C,
    # units per case, unit cost EUR, demand weight, season, weekly runs (weekday, Paris time, line)
    ("PRD-001", "SP", "Contoso Fresh Baby Spinach 200 g", "SUP-008", "Salad", "200 g", 6, 4, 10, 0.85, 1.2, "produce", [(d, "01:00", "FPP-L1") for d in MON_TO_SAT]),
    ("PRD-002", "ML", "Contoso Fresh Mixed Salad Leaves 150 g", "SUP-008", "Salad", "150 g", 6, 4, 10, 0.75, 1.1, "produce", [(d, "03:00", "FPP-L1") for d in MON_TO_SAT]),
    ("PRD-003", "CT", "Contoso Fresh Cherry Tomatoes 250 g", "SUP-008", "Vegetables", "250 g", 9, 8, 10, 0.95, 1.2, "produce", [(d, "01:00", "FPP-L2") for d in MON_TO_SAT]),
    ("PRD-004", "ST", "Contoso Fresh Strawberries 400 g", "SUP-008", "Fruit", "400 g", 5, 4, 8, 1.60, 1.0, "produce", [(d, "05:00", "FPP-L1") for d in MON_TO_SAT]),
    ("PRD-005", "AV", "Contoso Fresh Ripe Avocados 2 pack", "SUP-008", "Fruit", "2 pack", 7, 8, 12, 1.10, 0.9, "produce", [(d, "03:00", "FPP-L2") for d in MON_TO_SAT]),
    ("PRD-006", "HB", "Contoso Fresh Basil 30 g", "SUP-008", "Herbs", "30 g", 7, 8, 12, 0.55, 0.6, "produce", [(d, "05:00", "FPP-L2") for d in MON_TO_SAT]),
    ("PRD-007", "YG", "Contoso Fresh Greek Yogurt 500 g", "SUP-014", "Yogurt", "500 g", 28, 4, 12, 0.62, 1.1, "dairy", [(SUN, "22:00", "AVD-L2"), (TUE, "06:00", "AVD-L1"), (THU, "06:00", "AVD-L1")]),
    ("PRD-008", "NY", "Contoso Fresh Natural Yogurt 1 kg", "SUP-014", "Yogurt", "1 kg", 28, 4, 6, 0.95, 0.9, "dairy", [(MON, "06:00", "AVD-L2"), (WED, "06:00", "AVD-L2"), (FRI, "06:00", "AVD-L2")]),
    ("PRD-009", "SY", "Contoso Fresh Strawberry Yogurt 4 x 125 g", "SUP-014", "Yogurt", "4 x 125 g", 24, 4, 6, 0.70, 0.8, "dairy", [(MON, "06:00", "AVD-L1"), (WED, "14:00", "AVD-L1")]),
    ("PRD-010", "KF", "Contoso Fresh Kefir 1 L", "SUP-014", "Cultured dairy", "1 L", 21, 4, 6, 0.80, 0.5, "dairy", [(TUE, "14:00", "AVD-L2"), (FRI, "06:00", "AVD-L1")]),
    ("PRD-011", "FB", "Contoso Fresh Fromage Blanc 500 g", "SUP-014", "Cultured dairy", "500 g", 21, 4, 8, 0.85, 0.5, "dairy", [(THU, "14:00", "AVD-L1"), (SAT, "06:00", "AVD-L2")]),
    ("PRD-012", "SC", "Contoso Fresh Sour Cream 200 ml", "SUP-014", "Cultured dairy", "200 ml", 30, 4, 12, 0.45, 0.6, "dairy", [(MON, "14:00", "AVD-L2"), (FRI, "14:00", "AVD-L1")]),
    ("PRD-013", "LS", "Contoso Fresh Beef Lasagne 400 g", "SUP-021", "Frozen meals", "400 g", 180, -18, 8, 1.45, 0.6, "frozen", [(MON, "07:00", "NFF-L1")]),
    ("PRD-014", "VC", "Contoso Fresh Vegetable Curry 350 g", "SUP-021", "Frozen meals", "350 g", 180, -18, 8, 1.30, 0.4, "frozen", [(TUE, "07:00", "NFF-L1")]),
    ("PRD-015", "FP", "Contoso Fresh Fish Pie 400 g", "SUP-021", "Frozen meals", "400 g", 180, -18, 8, 1.70, 0.4, "frozen", [(WED, "07:00", "NFF-L1")]),
    ("PRD-016", "PZ", "Contoso Fresh Margherita Pizza 350 g", "SUP-021", "Frozen pizza", "350 g", 180, -18, 8, 1.10, 0.7, "frozen", [(MON, "07:00", "NFF-L2")]),
    ("PRD-017", "SR", "Contoso Fresh Vegetable Spring Rolls 12 pack", "SUP-021", "Frozen snacks", "12 pack", 180, -18, 8, 1.20, 0.3, "frozen", [(WED, "07:00", "NFF-L2")]),
    ("PRD-018", "CS", "Contoso Fresh Chicken Stir-Fry 500 g", "SUP-021", "Frozen meals", "500 g", 180, -18, 8, 1.90, 0.4, "frozen", [(THU, "07:00", "NFF-L1")]),
    ("PRD-019", "MZ", "Contoso Fresh Mozzarella 125 g", "SUP-027", "Cheese", "125 g", 21, 4, 12, 0.60, 0.8, "cheese", [(MON, "08:00", "WGC-L1"), (THU, "08:00", "WGC-L1")]),
    ("PRD-020", "CH", "Contoso Fresh Mature Cheddar 400 g", "SUP-027", "Cheese", "400 g", 120, 4, 8, 2.10, 0.4, "cheese", [(TUE, "08:00", "WGC-L2")]),
    ("PRD-021", "BR", "Contoso Fresh Brie 200 g", "SUP-027", "Cheese", "200 g", 35, 4, 12, 1.40, 0.4, "cheese", [(WED, "08:00", "WGC-L2")]),
    ("PRD-022", "PM", "Contoso Fresh Grated Parmesan 100 g", "SUP-027", "Cheese", "100 g", 60, 4, 12, 1.15, 0.3, "cheese", [(FRI, "08:00", "WGC-L2")]),
    ("PRD-023", "BT", "Contoso Fresh Unsalted Butter 250 g", "SUP-027", "Butter and cream", "250 g", 60, 4, 20, 1.05, 0.6, "cheese", [(TUE, "08:00", "WGC-L1"), (FRI, "08:00", "WGC-L1")]),
    ("PRD-024", "CM", "Contoso Fresh Double Cream 300 ml", "SUP-027", "Butter and cream", "300 ml", 14, 4, 12, 0.85, 0.5, "cheese", [(MON, "14:00", "WGC-L2"), (THU, "14:00", "WGC-L2")]),
]

# One-off runs outside the weekly schedule: lot code, line, day, Paris start time, minutes
EXTRA_RUNS = [
    ("MZ", "WGC-L1", date(2026, 9, 6), "23:30", 180),
    ("YG", "AVD-L2", date(2026, 9, 19), "23:00", 240),
    ("YG", "AVD-L1", date(2026, 9, 27), "23:00", 240),
    ("YG", "AVD-L2", date(2026, 10, 12), "02:00", 240),
]

RUN_MINUTES = {"produce": (90, 150), "dairy": (210, 270), "frozen": (300, 420), "cheese": (180, 300)}
SEASON = {  # demand by month, January first
    "produce": (0.80, 0.80, 0.90, 1.00, 1.15, 1.30, 1.35, 1.30, 1.10, 0.95, 0.85, 0.90),
    "dairy": (0.90, 0.90, 1.00, 1.00, 1.10, 1.20, 1.25, 1.20, 1.05, 1.00, 0.95, 0.95),
    "frozen": (1.20, 1.15, 1.05, 0.95, 0.90, 0.85, 0.80, 0.85, 0.95, 1.05, 1.15, 1.20),
    "cheese": (1.00, 0.95, 0.95, 1.00, 1.00, 0.95, 0.90, 0.90, 1.00, 1.05, 1.15, 1.35),
}
STORE_MONTH = (0.95, 0.95, 1.00, 1.00, 1.00, 1.00, 1.00, 0.95, 1.00, 1.00, 1.05, 1.20)

# Store cities and their approximate latitude and longitude
REGIONS = {
    "North": [("Lille", 50.63, 3.06), ("Roubaix", 50.69, 3.18), ("Tourcoing", 50.72, 3.16), ("Dunkerque", 51.03, 2.38),
              ("Calais", 50.95, 1.86), ("Boulogne-sur-Mer", 50.73, 1.61), ("Bethune", 50.53, 2.64), ("Lens", 50.43, 2.83),
              ("Douai", 50.37, 3.08), ("Valenciennes", 50.36, 3.52), ("Amiens", 49.89, 2.30), ("Beauvais", 49.43, 2.08),
              ("Rouen", 49.44, 1.10), ("Abbeville", 50.11, 1.83), ("Saint-Quentin", 49.85, 3.29), ("Cambrai", 50.18, 3.24)],
    "East": [("Reims", 49.26, 4.03), ("Epernay", 49.04, 3.96), ("Chalons-en-Champagne", 48.96, 4.36),
             ("Charleville-Mezieres", 49.77, 4.72), ("Sedan", 49.70, 4.94), ("Laon", 49.56, 3.62), ("Soissons", 49.38, 3.32),
             ("Troyes", 48.30, 4.08), ("Verdun", 49.16, 5.38), ("Metz", 49.12, 6.18), ("Nancy", 48.69, 6.18),
             ("Thionville", 49.36, 6.17), ("Bar-le-Duc", 48.77, 5.16), ("Saint-Dizier", 48.64, 4.95),
             ("Vitry-le-Francois", 48.72, 4.59), ("Rethel", 49.51, 4.37)],
    "Central": [("Paris", 48.86, 2.35), ("Versailles", 48.80, 2.13), ("Nanterre", 48.89, 2.21), ("Saint-Denis", 48.94, 2.36),
                ("Argenteuil", 48.95, 2.25), ("Montreuil", 48.86, 2.44), ("Meaux", 48.96, 2.88), ("Melun", 48.54, 2.66),
                ("Chartres", 48.45, 1.49), ("Orleans", 47.90, 1.91), ("Tours", 47.39, 0.69), ("Blois", 47.59, 1.33),
                ("Bourges", 47.08, 2.40), ("Evry", 48.63, 2.44), ("Fontainebleau", 48.40, 2.70), ("Rambouillet", 48.64, 1.83)],
    "West": [("Le Mans", 48.00, 0.20), ("Angers", 47.47, -0.55), ("Nantes", 47.22, -1.55), ("Rennes", 48.11, -1.68),
             ("Laval", 48.07, -0.77), ("Caen", 49.18, -0.37), ("Alencon", 48.43, 0.09), ("Poitiers", 46.58, 0.34),
             ("Niort", 46.32, -0.46), ("Cholet", 47.06, -0.88), ("Saumur", 47.26, -0.08), ("La Roche-sur-Yon", 46.67, -1.43),
             ("Saint-Nazaire", 47.27, -2.21), ("Lisieux", 49.15, 0.23), ("Chatellerault", 46.82, 0.55), ("Vendome", 47.79, 1.07)],
    "South": [("Lyon", 45.76, 4.84), ("Villeurbanne", 45.77, 4.88), ("Saint-Etienne", 45.44, 4.39), ("Grenoble", 45.19, 5.72),
              ("Valence", 44.93, 4.89), ("Chambery", 45.56, 5.92), ("Annecy", 45.90, 6.13), ("Vienne", 45.53, 4.87),
              ("Bourg-en-Bresse", 46.21, 5.23), ("Montelimar", 44.56, 4.75), ("Avignon", 43.95, 4.81), ("Nimes", 43.84, 4.36),
              ("Montpellier", 43.61, 3.88), ("Aix-en-Provence", 43.53, 5.45), ("Marseille", 43.30, 5.37),
              ("Clermont-Ferrand", 45.78, 3.08)],
}
DISTRIBUTION_CENTERS = [
    {"id": "DC-01", "name": "Contoso Fresh Laon DC", "city": "Laon", "region": "East", "opened": date(2009, 3, 2),
     "at": (49.60, 3.64), "serves": ("North", "East"), "fleet": {"CAR-01": 6, "CAR-02": 5, "CAR-04": 3}},
    {"id": "DC-02", "name": "Contoso Fresh Le Mans DC", "city": "Le Mans", "region": "West", "opened": date(2011, 6, 6),
     "at": (47.97, 0.16), "serves": ("Central", "West"), "fleet": {"CAR-01": 6, "CAR-02": 5, "CAR-03": 4}},
    {"id": "DC-03", "name": "Contoso Fresh Valence DC", "city": "Valence", "region": "South", "opened": date(2014, 9, 1),
     "at": (44.88, 4.88), "serves": ("South",), "fleet": {"CAR-01": 4, "CAR-03": 4, "CAR-04": 3}},
]
ROAD_MINUTES_PER_KM = 1.0  # straight-line km to road minutes, about 75 km/h on roads 1.25 times longer
UNLOAD_MINUTES = 30
CHANGEOVER_MINUTES = 30
CARRIERS = [
    ("CAR-01", "Contoso Fresh Fleet", "false"),
    ("CAR-02", "Wide World Logistics", "true"),
    ("CAR-03", "Relecloud Freight", "true"),
    ("CAR-04", "Proseware Transport", "true"),
]
REEFER_MODELS = ["Litware CoolRun 1200", "Litware CoolRun 1500 MT", "Lamna FrostLine 900"]

# Free text for AI Functions: (English, French)
DRIVER_NOTES = {
    "delivered": [
        ("Reefer alarm at {temp} C for {minutes} min, cleared before arrival.",
         "Alarme groupe froid à {temp} °C pendant {minutes} min, revenue à la normale avant l'arrivée."),
        ("Pallet wrap torn at loading, cases checked and accepted.",
         "Film de palette déchiré au chargement, colis vérifiés et acceptés."),
        ("Rear door seal icing up, reported to fleet maintenance.",
         "Joint de porte arrière givré, signalé à la maintenance."),
        ("Store dock busy, waited {minutes} min to unload.",
         "Quai du magasin occupé, attente de {minutes} min avant déchargement."),
    ],
    "late": [("Arrived {minutes} min late, heavy traffic on the ring road.",
              "Arrivée avec {minutes} min de retard, bouchons sur la rocade.")],
    "short": [("Short {cases} cs against the delivery note.",
               "Il manque {cases} colis par rapport au bon de livraison.")],
    "refused": [("Refused by store: {cases} cs, product at {temp} C on arrival.",
                 "Refusé par le magasin : {cases} colis, produit à {temp} °C à l'arrivée.")],
}
QA_NOTES = [
    ("Fill weight check: {count} of 50 units under target, within tolerance.",
     "Contrôle poids : {count} unités sur 50 sous la cible, dans la tolérance."),
    ("Cooling tunnel outlet at {temp} C for {minutes} min, product held and released after retest.",
     "Sortie du tunnel de refroidissement à {temp} °C pendant {minutes} min, lot bloqué puis libéré après contrôle."),
    ("Labels printed with the wrong date and reprinted before release.",
     "Étiquettes imprimées avec une date erronée, réimprimées avant libération."),
]
NIGHT_QA_NOTES = [
    ("Night crew one operator short, CIP cycle on AVD-L2 cut by {minutes} min.",
     "Équipe de nuit en sous-effectif, cycle NEP de la ligne AVD-L2 écourté de {minutes} min."),
    ("Environmental swab near the AVD-L2 filler sent to the lab, result pending at release.",
     "Prélèvement d'environnement près de la remplisseuse AVD-L2 envoyé au labo, résultat en attente à la libération."),
]


class Rng:
    """Draws only from random(), which gives the same sequence on every Python 3 version."""

    def __init__(self, stream):
        self.random = random.Random(f"{SEED}-{stream}").random

    def chance(self, p):
        return self.random() < p

    def between(self, low, high):
        return low + int(self.random() * (high - low + 1))

    def uniform(self, low, high):
        return low + (high - low) * self.random()

    def pick(self, items):
        return items[int(self.random() * len(items))]

    def sample(self, items, k):
        pool = list(items)
        return [pool.pop(int(self.random() * len(pool))) for _ in range(k)]


def paris(day, hhmm="00:00", minutes=0):
    """A Paris wall-clock time on a day, plus minutes, returned in UTC."""
    hour, minute = map(int, hhmm.split(":"))
    local = datetime(day.year, day.month, day.day, hour, minute, tzinfo=PARIS) + timedelta(minutes=minutes)
    return local.astimezone(UTC)


def iso(moment):
    return moment.strftime("%Y-%m-%dT%H:%M:%SZ")


def days(first, last):
    for n in range((last - first).days + 1):
        yield first + timedelta(days=n)


def km(a, b):
    """Straight-line distance between two (latitude, longitude) points in France, without trigonometry."""
    dy, dx = (b[0] - a[0]) * 111.2, (b[1] - a[1]) * 76.5
    return math.sqrt(dx * dx + dy * dy)


def drive(a, b):
    return int(km(a, b) * ROAD_MINUTES_PER_KM) + 10


def heading(a, b):
    """Compass order from 0 to 4, counterclockwise from west."""
    dy, dx = b[0] - a[0], (b[1] - a[1]) * 0.69
    if dx == 0 and dy == 0:
        return 0.0
    if dy >= 0:
        turn = dy / (dx + dy) if dx >= 0 else 1 - dx / (dy - dx)
    else:
        turn = 2 - dy / (-dx - dy) if dx < 0 else 3 + dx / (dx - dy)
    return (turn + 2) % 4


def note(rng, templates, **fixed):
    english, french = rng.pick(templates)
    in_french = rng.chance(FRENCH_NOTE_SHARE)
    temp = f"{rng.uniform(4.6, 9.8):.1f}"
    values = {"temp": temp.replace(".", ",") if in_french else temp, "minutes": rng.between(15, 75),
              "count": rng.between(1, 6), "cases": 1}
    values.update(fixed)
    return (french if in_french else english).format(**values)


def build_locations(rng):
    stores = []
    for region, cities in REGIONS.items():
        late_openers = rng.sample(range(len(cities)), len(cities) - STORES_AT_START // len(REGIONS))
        for i, (city, lat, lon) in enumerate(cities):
            if i in late_openers:
                opened = FIRST_DAY + timedelta(days=rng.between(60, 1800))
            else:
                opened = date(2006, 1, 1) + timedelta(days=rng.between(0, 5700))
            stores.append({"id": f"STR-{len(stores) + 1:03d}", "name": f"Contoso Fresh {city}", "city": city,
                           "region": region, "opened": opened, "at": (lat, lon), "size": rng.uniform(0.75, 1.3)})
    for dc in DISTRIBUTION_CENTERS:
        for store in stores:
            if store["region"] in dc["serves"]:
                store["dc"] = dc["id"]
    return stores


def build_trucks(rng):
    trucks = []
    for dc in DISTRIBUTION_CENTERS:
        for carrier_id, count in dc["fleet"].items():
            for _ in range(count):
                trucks.append({"id": f"TRK-{len(trucks) + 1:03d}", "carrier": carrier_id, "dc": dc["id"],
                               "reefer": rng.pick(REEFER_MODELS)})
    return trucks


def build_lots(rng, products):
    recent_sundays = [RECALL_DAY - timedelta(weeks=k) for k in range(1, 13)]
    after_midnight = rng.sample(recent_sundays, 5)
    by_code = {p["code"]: p for p in products}
    runs = []
    for day in days(FIRST_DAY, AS_OF.date()):
        for p in products:
            for weekday, hhmm, line in p["runs"]:
                if weekday != day.weekday():
                    continue
                minutes = rng.between(*RUN_MINUTES[p["season"]])
                night_run = (p["code"], weekday) == ("YG", SUN)
                if not night_run:
                    offset = rng.between(0, 40)
                elif day in (RECALL_DAY, RECALL_DAY - timedelta(weeks=13)):
                    offset, minutes = 0, 240
                elif day in after_midnight:
                    offset = rng.between(125, 225)
                elif day in recent_sundays:
                    offset = rng.between(5, 110)
                else:
                    offset = rng.between(0, 225)  # the Sunday night run starts any time from 22:00 to 01:45
                related = night_run and (day == RECALL_DAY or day in recent_sundays)
                runs.append((p, line, day, paris(day, hhmm, offset), minutes, related))
    for code, line, day, hhmm, minutes in EXTRA_RUNS:
        runs.append((by_code[code], line, day, paris(day, hhmm), minutes, False))
    line_free, scheduled = {}, []
    for p, line, day, start, minutes, related in sorted(runs, key=lambda r: (r[1], r[3], r[0]["id"])):
        previous, free_at = line_free.get(line, (p, start))
        start = max(start, free_at + timedelta(minutes=0 if previous is p else CHANGEOVER_MINUTES))
        line_free[line] = (p, start + timedelta(minutes=minutes))
        scheduled.append((p, line, day, start, minutes, related))
    runs = scheduled

    lots = []
    for p in products:
        own = sorted((r for r in runs if r[0] is p), key=lambda r: r[3])
        own = [r for r in own if r[3] + timedelta(minutes=r[4]) <= AS_OF]
        if p["code"] == "YG":
            first_number = 2291 - next(i for i, r in enumerate(own) if r[3] == paris(RECALL_DAY, "22:00"))
        else:
            first_number = rng.between(1000, 6000)
        for i, (_, line, day, start, minutes, related) in enumerate(own):
            years = (start - paris(FIRST_DAY)).days / 365.25
            cost = p["unit_cost"] * (1 + COST_GROWTH_PER_YEAR * years) * rng.uniform(0.98, 1.02)
            lot_id = f"{p['code']}-{first_number + i:04d}"
            if lot_id == "YG-2291":
                qa_note = note(rng, NIGHT_QA_NOTES[1:])
            elif related and rng.chance(0.4):
                qa_note = note(rng, NIGHT_QA_NOTES[:1])
            elif rng.chance(QA_NOTE_RATE):
                qa_note = note(rng, QA_NOTES)
            else:
                qa_note = ""
            lots.append({"id": lot_id, "product": p, "line": line, "day": day, "start": start,
                         "end": start + timedelta(minutes=minutes),
                         "expires": start.astimezone(PARIS).date() + timedelta(days=p["shelf_life"]),
                         "cost": round(cost, 2),
                         "qa_note": qa_note, "related": related})
    return lots


def delivery_days():
    return [d for d in days(FIRST_DAY, LAST_DELIVERY_DAY)
            if d.weekday() != SUN and (d.month, d.day) not in CLOSED_DAYS]


def nearest_first(start, members):
    left, here, order = list(members), start, []
    while left:
        stop = min(left, key=lambda s: (km(here, s["at"]), s["id"]))
        left.remove(stop)
        order.append(stop)
        here = stop["at"]
    return order


def build_routes(stores, trucks):
    """Each DC sends one truck out on each of its fixed multi-drop routes. Trucks rotate between routes day by day."""
    networks = []
    for dc in DISTRIBUTION_CENTERS:
        fleet = [t for t in trucks if t["dc"] == dc["id"]]
        served = sorted((s for s in stores if s["dc"] == dc["id"]), key=lambda s: (heading(dc["at"], s["at"]), s["id"]))
        n = len(fleet)
        groups = [served[i * len(served) // n:(i + 1) * len(served) // n] for i in range(n)]
        networks.append((dc, fleet, [nearest_first(dc["at"], g) for g in groups]))
    cache = {}

    def routes(day):
        """Returns each store's delivery that day, and the time each truck is back at its DC."""
        if day in cache:
            return cache[day]
        rng = Rng(f"routes-{day.isoformat()}")
        plan, back = {}, {}
        for dc, fleet, groups in networks:
            for k, members in enumerate(groups):
                truck = fleet[(k + day.toordinal()) % len(fleet)]
                departure = paris(day, "04:00", 20 * k + rng.between(0, 10))
                late = rng.between(30, 120) if rng.chance(LATE_SHIPMENT_RATE) else 0
                here, clock = dc["at"], departure + timedelta(minutes=late)
                for s in members:
                    if s["opened"] > day:
                        continue
                    clock += timedelta(minutes=drive(here, s["at"]))
                    plan[s["id"]] = {"truck": truck, "shipped": departure, "delivered": clock, "late": late}
                    clock += timedelta(minutes=UNLOAD_MINUTES)
                    here = s["at"]
                back[truck["id"]] = clock + timedelta(minutes=drive(here, dc["at"]))
        cache.clear()
        cache[day] = plan, back
        return plan, back

    return routes


def spare_truck(rng, trucks, routes, day, dc_id, carrier_id, leaves):
    _, back = routes(day)
    return rng.pick([t for t in trucks if t["dc"] == dc_id and t["carrier"] == carrier_id
                     and back[t["id"]] + timedelta(minutes=30) <= leaves])


def plan_special_deliveries(rng, stores, trucks, lots, routes):
    related = sorted((lot for lot in lots if lot["related"]), key=lambda lot: lot["start"])
    main_lot = related[-1]
    monday = LAST_DELIVERY_DAY
    top_up_day = date(2026, 9, 8)

    def ready(store, day, lot):
        route = routes(day)[0].get(store["id"])
        return route is not None and route["shipped"] >= lot["end"] + timedelta(hours=4)

    logged, actual = rng.sample([s for s in stores if s["dc"] == "DC-02" and s["opened"] <= top_up_day], 2)
    held_back = {logged["id"], actual["id"]}
    options = [s for s in stores if s["id"] not in held_back and ready(s, monday, main_lot)]
    main_stores = (rng.sample([s for s in options if s["region"] == "North"], 7)
                   + rng.sample([s for s in options if s["region"] == "East"], 5))
    main_ids = {s["id"] for s in main_stores}
    pinned = {}
    for s in main_stores:
        pinned.setdefault((monday, s["id"]), []).append(main_lot)
    for lot in related[:-1]:
        day = lot["day"] + timedelta(days=1)
        for s in rng.sample([s for s in stores if s["id"] not in held_back and s["opened"] <= day], 4):
            ship_day = day if ready(s, day, lot) else day + timedelta(days=1)
            pinned.setdefault((ship_day, s["id"]), []).append(lot)
    reached = main_ids | held_back | {store_id for _, store_id in pinned}
    late_store = rng.pick([s for s in stores if s["region"] == "North" and s["id"] not in reached])
    late_leaves = paris(monday, "14:25", rng.between(0, 10))
    top_up_leaves = paris(top_up_day, "13:00", rng.between(0, 10))
    return {
        "main_lot": main_lot, "main_stores": main_stores, "pinned": pinned,
        "late": {"store": late_store, "shipped": late_leaves,
                 "truck": spare_truck(rng, trucks, routes, monday, "DC-01", "CAR-02", late_leaves)},
        "top_up": {"day": top_up_day, "logged": logged, "actual": actual,
                   "lot": next(lot for lot in related if lot["day"] == date(2026, 9, 6)),
                   "shipped": top_up_leaves, "cases": rng.between(3, 8),
                   "truck": spare_truck(rng, trucks, routes, top_up_day, "DC-02", "CAR-03", top_up_leaves)},
        "duplicate_days": rng.sample([d for d in delivery_days() if date(2026, 10, 5) <= d <= date(2026, 10, 10)], 2),
    }


def pick_recent(rng, candidates):
    r = rng.random()
    return candidates[min(0 if r < 0.6 else 1 if r < 0.9 else 2, len(candidates) - 1)]


def line_row(line_no, shipped, delivered, store, truck, lot, cases, status, text):
    p = lot["product"]
    cost = cases * p["units_per_case"] * lot["cost"]
    return [line_no, iso(shipped), iso(delivered), store["dc"], store["id"], truck["carrier"], truck["id"],
            lot["id"], p["id"], p["supplier"], cases, f"{cost:.2f}", status, text]


def write_shipments(rng, writer, stores, products, lots, routes, special):
    stock = {p["id"]: sorted((lot for lot in lots if lot["product"] is p and not lot["related"]),
                             key=lambda lot: (lot["end"], lot["id"])) for p in products}
    pointer = {product_id: 0 for product_id in stock}
    total_demand = sum(p["demand"] for p in products)
    duplicate_rng = Rng("duplicates")
    dc_at = {dc["id"]: dc["at"] for dc in DISTRIBUTION_CENTERS}
    last, next_number = {}, 100001
    result = {"shipments": 0, "lines": 0, "notes": 0, "duplicates": [], "top_up": None, "cases": {}}

    def emit(rows, real=True):
        nonlocal next_number
        shipment_id = f"SHP-{next_number:06d}"
        next_number += 1
        for row in rows:
            writer.writerow([shipment_id] + row)
            if row[12] != "refused":
                last[(row[4], row[8])] = (row[7], row[10])
            if real:
                result["cases"][row[7]] = result["cases"].get(row[7], 0) + row[10]
            result["notes"] += row[13] != ""
        result["shipments"] += 1
        result["lines"] += len(rows)
        return shipment_id

    for day in delivery_days():
        midnight, month = paris(day), day.month - 1
        ready = {}
        for p in products:
            seq, i = stock[p["id"]], pointer[p["id"]]
            while i < len(seq) and seq[i]["end"] <= midnight:
                i += 1
            pointer[p["id"]] = i
            ready[p["id"]] = [lot for lot in reversed(seq[max(0, i - 3):i]) if lot["expires"] > day]
        plan, _ = routes(day)
        batch = []
        for s in stores:
            route = plan.get(s["id"])
            if route is None:
                continue
            pins = special["pinned"].get((day, s["id"]), [])
            lines = []
            for p in products:
                pinned_here = [lot for lot in pins if lot["product"] is p]
                if pinned_here:
                    lines.extend((lot, True) for lot in pinned_here)
                    continue
                rate = (LINES_PER_SHIPMENT * p["demand"] / total_demand * SEASON[p["season"]][month]
                        * STORE_MONTH[month] * s["size"])
                if rng.chance(min(rate, 0.95)) and ready[p["id"]]:
                    lines.append((pick_recent(rng, ready[p["id"]]), False))
            rows = []
            for line_no, (lot, is_pinned) in enumerate(lines, 1):
                p = lot["product"]
                cases = max(1, int(p["demand"] * s["size"] * SEASON[p["season"]][month] * rng.uniform(1.5, 4.5) + 0.5))
                status, text = ("late" if route["late"] else "delivered"), ""
                if is_pinned:
                    if rng.chance(0.5):
                        if lot is not special["main_lot"] and rng.chance(0.3):
                            status, text = "refused", note(rng, DRIVER_NOTES["refused"], cases=cases)
                        else:
                            text = note(rng, DRIVER_NOTES["delivered"][:1])
                else:
                    r = rng.random()
                    if r < REFUSED_LINE_RATE:
                        status = "refused"
                    elif r < REFUSED_LINE_RATE + SHORT_LINE_RATE:
                        status = "short"
                    if status == "refused" and rng.chance(ISSUE_NOTE_RATE):
                        text = note(rng, DRIVER_NOTES["refused"], cases=cases)
                    elif status == "short" and rng.chance(ISSUE_NOTE_RATE):
                        text = note(rng, DRIVER_NOTES["short"], cases=rng.between(1, max(1, cases // 2)))
                    elif status == "late" and line_no == 1 and rng.chance(ISSUE_NOTE_RATE):
                        text = note(rng, DRIVER_NOTES["late"], minutes=route["late"])
                    elif status in ("delivered", "late") and rng.chance(ROUTINE_NOTE_RATE):
                        text = note(rng, DRIVER_NOTES["delivered"])
                rows.append(line_row(line_no, route["shipped"], route["delivered"], s, route["truck"],
                                     lot, cases, status, text))
            if rows:
                batch.append((rows, bool(pins), route["truck"]))
        for rows, _, _ in batch:
            emit(rows)
        top_up = special["top_up"]
        if day == top_up["day"]:
            delivered = top_up["shipped"] + timedelta(minutes=drive(dc_at["DC-02"], top_up["actual"]["at"]))
            row = line_row(1, top_up["shipped"], delivered, top_up["logged"], top_up["truck"], top_up["lot"],
                           top_up["cases"], "delivered", "")
            result["top_up"] = [emit([row])] + row
        if day in special["duplicate_days"]:
            options = [rows for rows, has_pins, truck in batch
                       if truck["carrier"] == "CAR-02" and len(rows) == LINES_PER_SHIPMENT and not has_pins]
            rows = duplicate_rng.pick(options)
            shipment_id = emit(rows, real=False)
            result["duplicates"].append([[shipment_id] + row for row in rows])

    late = special["late"]
    store, day = late["store"], LAST_DELIVERY_DAY
    delivered = late["shipped"] + timedelta(minutes=drive(dc_at["DC-01"], store["at"]))
    by_code = {lot["id"][:2]: lot for lot in lots if lot["expires"] > day and lot["end"] <= paris(day)
               and not lot["related"] and lot["id"][:2] in ("NY", "KF")}
    late_rows = [line_row(n, late["shipped"], delivered, store, late["truck"], lot, rng.between(2, 6), "delivered", "")
                 for n, lot in enumerate([special["main_lot"], by_code["NY"], by_code["KF"]], 1)]
    late_id = f"SHP-{next_number:06d}"
    result["late_rows"] = [[late_id] + row for row in late_rows]
    for row in late_rows:
        last[(store["id"], row[8])] = (row[7], row[10])
        result["cases"][row[7]] = result["cases"].get(row[7], 0) + row[10]
    result["last"] = last
    return result


def size_lots(rng, lots, shipped_cases):
    """Suppliers sell each run to other retailers too, so a run is sized from its product's recent Contoso volume."""
    by_product = {}
    for lot in lots:
        by_product.setdefault(lot["product"]["id"], []).append(lot)
    for runs in by_product.values():
        runs.sort(key=lambda lot: lot["start"])
        volumes = [(lot["start"], shipped_cases[lot["id"]]) for lot in runs
                   if not lot["related"] and shipped_cases.get(lot["id"])]
        starts = [start for start, _ in volumes]
        for lot in runs:
            window = sorted(c for _, c in volumes[bisect.bisect_left(starts, lot["start"] - timedelta(days=42)):
                                                  bisect.bisect_right(starts, lot["start"] + timedelta(days=42))])
            typical = window[len(window) // 2] if window else 40
            cases = shipped_cases.get(lot["id"], 0)
            planned = int(typical * rng.uniform(1.15, 1.6))
            needed = cases + max(2, int(cases * rng.uniform(0.03, 0.12)))
            lot["units"] = max(planned, needed) * lot["product"]["units_per_case"]


def build_inventory(rng, last, lots, special):
    by_id = {lot["id"]: lot for lot in lots}
    main_id = special["main_lot"]["id"]
    holders = [s["id"] for s in special["main_stores"][:8]] + [special["late"]["store"]["id"]]
    rows = []
    for (store_id, product_id), (lot_id, cases) in sorted(last.items()):
        lot = by_id[lot_id]
        keep = store_id in holders if lot_id == main_id else lot["expires"] >= AS_OF_DAY and rng.chance(0.8)
        if keep:
            units = max(1, int(cases * lot["product"]["units_per_case"] * rng.uniform(0.15, 0.9)))
            counted = paris(LAST_DELIVERY_DAY, "21:00", rng.between(0, 150))
            rows.append([store_id, lot_id, units, iso(counted)])
    return rows


def write_csv(path, header, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, lineterminator="\n")
        writer.writerow(header)
        writer.writerows(rows)
    return len(rows)


SHIPMENT_COLUMNS = ["shipment_id", "line_no", "shipped_at_utc", "delivered_at_utc", "origin_dc_id", "store_id",
                    "carrier_id", "truck_id", "lot_id", "product_id", "supplier_id", "cases", "supplier_cost_eur",
                    "status", "driver_note"]
LOT_COLUMNS = ["lot_id", "product_id", "supplier_id", "production_line", "production_start_utc",
               "production_end_utc", "expires_on", "units_produced", "supplier_unit_cost_eur"]


def lot_row(lot):
    p = lot["product"]
    return [lot["id"], p["id"], p["supplier"], lot["line"], iso(lot["start"]), iso(lot["end"]),
            lot["expires"].isoformat(), lot["units"], f"{lot['cost']:.2f}"]


class NoFile:
    def writerow(self, row):
        pass


def generate(reference_dir, history_dir=None):
    products = [dict(zip(PRODUCT_FIELDS, row)) for row in PRODUCTS]
    stores = build_locations(Rng("locations"))
    trucks = build_trucks(Rng("trucks"))
    lots = build_lots(Rng("lots"), products)
    routes = build_routes(stores, trucks)
    special = plan_special_deliveries(Rng("recall"), stores, trucks, lots, routes)

    history = Path(history_dir) if history_dir else None
    handle = None
    if history:
        history.mkdir(parents=True, exist_ok=True)
        handle = open(history / "shipment_history.csv", "w", newline="", encoding="utf-8")
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerow(SHIPMENT_COLUMNS)
    else:
        writer = NoFile()
    try:
        shipped = write_shipments(Rng("shipments"), writer, stores, products, lots, routes, special)
    finally:
        if handle:
            handle.close()
    size_lots(Rng("production"), lots, shipped["cases"])

    reference = Path(reference_dir)
    dcs = [[dc["id"], dc["name"], "dc", dc["city"], dc["region"], dc["opened"].isoformat()] for dc in DISTRIBUTION_CENTERS]
    counts = {
        "suppliers": write_csv(reference / "suppliers.csv", ["supplier_id", "supplier_name", "category", "country"],
                               [list(s) for s in SUPPLIERS]),
        "products": write_csv(reference / "products.csv",
                              ["product_id", "product_name", "supplier_id", "category", "pack_size",
                               "shelf_life_days", "max_storage_c"],
                              [[p["id"], p["name"], p["supplier"], p["category"], p["pack_size"], p["shelf_life"],
                                p["max_c"]] for p in products]),
        "locations": write_csv(reference / "locations.csv",
                               ["location_id", "location_name", "location_type", "city", "region", "opened_on"],
                               dcs + [[s["id"], s["name"], "store", s["city"], s["region"], s["opened"].isoformat()]
                                      for s in stores]),
        "carriers": write_csv(reference / "carriers.csv", ["carrier_id", "carrier_name", "is_logistics_partner"],
                              [list(c) for c in CARRIERS]),
        "trucks": write_csv(reference / "trucks.csv", ["truck_id", "carrier_id", "home_dc_id", "reefer_model"],
                            [[t["id"], t["carrier"], t["dc"], t["reefer"]] for t in trucks]),
        "lots": write_csv(reference / "lots.csv", LOT_COLUMNS,
                          [lot_row(lot) for lot in sorted(lots, key=lambda lot: (lot["start"], lot["id"]))
                           if lot["expires"] >= AS_OF_DAY]),
        "store_inventory": write_csv(reference / "store_inventory.csv",
                                     ["location_id", "lot_id", "units_on_hand", "last_counted_utc"],
                                     build_inventory(Rng("inventory"), shipped["last"], lots, special)),
    }
    print(f"Wrote {len(counts)} tables to {reference}")
    if history:
        top_up = shipped["top_up"]
        corrected = top_up[:5] + [special["top_up"]["actual"]["id"]] + top_up[6:]
        counts["lot_history"] = write_csv(
            history / "lot_history.csv", LOT_COLUMNS + ["qa_note"],
            [lot_row(lot) + [lot["qa_note"]] for lot in sorted(lots, key=lambda lot: (lot["start"], lot["id"]))])
        counts["shipment_history"] = shipped["lines"]
        counts["corrections"] = write_csv(
            history / "corrections_2026-10-13.csv", ["action"] + SHIPMENT_COLUMNS,
            [["insert"] + row for row in shipped["late_rows"]] + [["update"] + corrected]
            + [["delete"] + row for rows in shipped["duplicates"] for row in rows])
        print(f"Wrote lot_history, shipment_history ({shipped['shipments']:,} shipments) "
              f"and the correction file to {history}")
    for table, rows in counts.items():
        print(f"  {table}: {rows:,} rows")
    return counts


def main():
    if importlib.util.find_spec("notebookutils"):
        files = Path("/lakehouse/default/Files")
        if not files.exists():
            raise RuntimeError("Add a default lakehouse to this notebook, then run this cell again.")
        generate(files / "reference", files / "history")
    else:
        here = Path.cwd().resolve()
        repo = next((d for d in [here, *here.parents] if (d / "data" / "sharepoint").is_dir()), None)
        if repo is None:
            raise RuntimeError("Run this cell from inside the PREL41 repository.")
        generate(repo / "data" / "reference")


if __name__ == "__main__":
    main()

## Exercise 2.2: Explore current state through OneLake

### Cell 2: Start Spark

Sets up two Iceberg catalogs: `writecat` writes to the lakehouse **Tables** folder, and `readcat` reads through the OneLake Iceberg REST catalog.

In [ ]:
ctx = notebookutils.runtime.context
workspace_id = ctx.get("defaultLakehouseWorkspaceId")
lakehouse_id = ctx.get("defaultLakehouseId")
if not workspace_id or not lakehouse_id:
    raise RuntimeError("The notebook has no lakehouse. Add the lakehouse in the Explorer pane, then run this cell again.")

try:
    iceberg_version = spark._jvm.org.apache.iceberg.IcebergBuild.version()
except Exception:
    raise RuntimeError(
        "Spark can't find Iceberg, which comes with Fabric Runtime 2.0. Check that the "
        "notebook runs on Runtime 2.0, then stop the session and run this cell again."
    ) from None

write_warehouse = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables"
read_warehouse = f"{workspace_id}/{lakehouse_id}"

# Standard Iceberg catalog settings: https://iceberg.apache.org/docs/latest/spark-configuration/
settings = {
    # writecat writes Iceberg files straight to the lakehouse Tables folder.
    "spark.sql.catalog.writecat": "org.apache.iceberg.spark.SparkCatalog",
    "spark.sql.catalog.writecat.type": "hadoop",
    "spark.sql.catalog.writecat.warehouse": write_warehouse,
    "spark.sql.catalog.writecat.io-impl": "org.apache.iceberg.hadoop.HadoopFileIO",
    # readcat reads through the OneLake Iceberg REST catalog, as any other engine would.
    "spark.sql.catalog.readcat": "org.apache.iceberg.spark.SparkCatalog",
    "spark.sql.catalog.readcat.type": "rest",
    "spark.sql.catalog.readcat.uri": "https://onelake.table.fabric.microsoft.com/iceberg",
    "spark.sql.catalog.readcat.warehouse": read_warehouse,
    "spark.sql.catalog.readcat.token": notebookutils.credentials.getToken("storage"),
    "spark.sql.catalog.readcat.io-impl": "org.apache.iceberg.hadoop.HadoopFileIO",
    # No catalog cache, so each read asks OneLake for the current metadata.
    "spark.sql.catalog.readcat.cache-enabled": "false",
    "spark.sql.session.timeZone": "UTC",
}
for key, value in settings.items():
    spark.conf.set(key, value)

print(f"Spark {spark.version}, Iceberg {iceberg_version}")
print(f"writecat warehouse: {write_warehouse}")
print(f"readcat warehouse:  {read_warehouse}")

### Cell 3: Build the current-state tables

Writes the 7 CSV files in **Files/reference** as Iceberg tables in `dbo`. A rerun keeps the tables it already built.

In [ ]:
if spark.conf.get("spark.sql.catalog.writecat", None) is None:
    raise RuntimeError("Run cell 2 first. It sets up the Iceberg catalogs.")

OWNER = "prel41-module-2"
warehouse = spark.conf.get("spark.sql.catalog.writecat.warehouse")


def create_once(table, source, partitioned_by=""):
    """Write source to writecat.<table>, unless this lab already wrote it. Never overwrite anything else."""
    folder = table.replace(".", "/")
    if notebookutils.fs.exists(f"{warehouse}/{folder}"):
        try:
            owner = spark.sql(f"SHOW TBLPROPERTIES writecat.{table} ('contoso.owner')").first()["value"]
        except Exception:
            owner = None
        if owner != OWNER:
            raise RuntimeError(f"Tables/{folder} already holds a table this lab didn't create, so the cell stopped without changing it.")
        rows = spark.table(f"writecat.{table}").count()
        print(f"{table}: already exists, {rows:,} rows")
        return
    source.createOrReplaceTempView("source")
    spark.sql(f"""
        CREATE TABLE writecat.{table}
        USING iceberg
        {partitioned_by}
        TBLPROPERTIES ('format-version' = '2', 'contoso.owner' = '{OWNER}')
        AS SELECT * FROM source
    """)
    rows = spark.table(f"writecat.{table}").count()
    print(f"{table}: {rows:,} rows")


tables = {
    "suppliers": "supplier_id STRING, supplier_name STRING, category STRING, country STRING",
    "products": "product_id STRING, product_name STRING, supplier_id STRING, category STRING, pack_size STRING, shelf_life_days INT, max_storage_c INT",
    "locations": "location_id STRING, location_name STRING, location_type STRING, city STRING, region STRING, opened_on DATE",
    "carriers": "carrier_id STRING, carrier_name STRING, is_logistics_partner BOOLEAN",
    "trucks": "truck_id STRING, carrier_id STRING, home_dc_id STRING, reefer_model STRING",
    "lots": "lot_id STRING, product_id STRING, supplier_id STRING, production_line STRING, production_start_utc TIMESTAMP, production_end_utc TIMESTAMP, expires_on DATE, units_produced INT, supplier_unit_cost_eur DECIMAL(10,2)",
    "store_inventory": "location_id STRING, lot_id STRING, units_on_hand INT, last_counted_utc TIMESTAMP",
}

for name, ddl in tables.items():
    csv = spark.read.schema(ddl).options(header=True, escape='"', mode="FAILFAST", timestampFormat="yyyy-MM-dd'T'HH:mm:ssXXX").csv(f"Files/reference/{name}.csv")
    create_once(f"dbo.{name}", csv)

### Cell 4: List the tables through OneLake

Waits until the OneLake REST catalog shows all 7 tables.

In [ ]:
import time

if spark.conf.get("spark.sql.catalog.readcat", None) is None:
    raise RuntimeError("Run cell 2 first. It sets up the Iceberg catalogs.")

expected = {"suppliers", "products", "locations", "carriers", "trucks", "lots", "store_inventory"}
deadline = time.monotonic() + 180
while True:
    tables = spark.sql("SHOW TABLES IN readcat.dbo")
    missing = expected - {row["tableName"] for row in tables.collect()}
    if not missing:
        break
    if time.monotonic() > deadline:
        names = ", ".join(f"dbo.{name}" for name in sorted(missing))
        raise TimeoutError(f"After 3 minutes, OneLake still doesn't show {names}. Run this cell again.")
    time.sleep(10)

display(tables)

### Cell 5: Read lot YG-2291

In [ ]:
if spark.conf.get("spark.sql.catalog.readcat", None) is None:
    raise RuntimeError("Run cell 2 first. It sets up the Iceberg catalogs.")

display(spark.sql("SELECT * FROM readcat.dbo.lots WHERE lot_id = 'YG-2291'"))

### Cell 6: Describe the lots table

Shows the table provider and where the files live.

In [ ]:
if spark.conf.get("spark.sql.catalog.readcat", None) is None:
    raise RuntimeError("Run cell 2 first. It sets up the Iceberg catalogs.")

display(spark.sql("DESCRIBE TABLE EXTENDED readcat.dbo.lots"))

### Cell 7: Find the stores that hold YG-2291

The question from Exercise 1.4, answered from Iceberg tables.

In [ ]:
if spark.conf.get("spark.sql.catalog.readcat", None) is None:
    raise RuntimeError("Run cell 2 first. It sets up the Iceberg catalogs.")

stores = spark.sql("""
    SELECT i.location_id AS store_id, l.location_name, l.city, l.region, i.units_on_hand
    FROM readcat.dbo.store_inventory AS i
    JOIN readcat.dbo.locations AS l ON l.location_id = i.location_id
    WHERE i.lot_id = 'YG-2291' AND l.location_type = 'store'
    ORDER BY store_id
""")
print(f"{stores.count()} stores hold YG-2291")
display(stores)

## Exercise 2.3: Write shipment history as Iceberg

### Cell 8: Write the history tables

Creates the `history` schema, then writes `lot_history` and `shipment_history`. `shipment_history` is partitioned by month. A rerun keeps the tables it already built.

In [ ]:
if spark.conf.get("spark.sql.catalog.writecat", None) is None:
    raise RuntimeError("Run cell 2 first. It sets up the Iceberg catalogs.")

# The lakehouse creates the schema, and writecat writes into its Tables/history folder.
spark.sql("CREATE SCHEMA IF NOT EXISTS history")

OWNER = "prel41-module-2"
warehouse = spark.conf.get("spark.sql.catalog.writecat.warehouse")


def create_once(table, source, partitioned_by=""):
    """Write source to writecat.<table>, unless this lab already wrote it. Never overwrite anything else."""
    folder = table.replace(".", "/")
    if notebookutils.fs.exists(f"{warehouse}/{folder}"):
        try:
            owner = spark.sql(f"SHOW TBLPROPERTIES writecat.{table} ('contoso.owner')").first()["value"]
        except Exception:
            owner = None
        if owner != OWNER:
            raise RuntimeError(f"Tables/{folder} already holds a table this lab didn't create, so the cell stopped without changing it.")
        rows = spark.table(f"writecat.{table}").count()
        print(f"{table}: already exists, {rows:,} rows")
        return
    source.createOrReplaceTempView("source")
    spark.sql(f"""
        CREATE TABLE writecat.{table}
        USING iceberg
        {partitioned_by}
        TBLPROPERTIES ('format-version' = '2', 'contoso.owner' = '{OWNER}')
        AS SELECT * FROM source
    """)
    rows = spark.table(f"writecat.{table}").count()
    print(f"{table}: {rows:,} rows")


tables = {
    "lot_history": ("lot_id STRING, product_id STRING, supplier_id STRING, production_line STRING, production_start_utc TIMESTAMP, production_end_utc TIMESTAMP, expires_on DATE, units_produced INT, supplier_unit_cost_eur DECIMAL(10,2), qa_note STRING", ""),
    "shipment_history": ("shipment_id STRING, line_no INT, shipped_at_utc TIMESTAMP, delivered_at_utc TIMESTAMP, origin_dc_id STRING, store_id STRING, carrier_id STRING, truck_id STRING, lot_id STRING, product_id STRING, supplier_id STRING, cases INT, supplier_cost_eur DECIMAL(10,2), status STRING, driver_note STRING", "PARTITIONED BY (months(shipped_at_utc))"),
}

for name, (ddl, partitioned_by) in tables.items():
    csv = spark.read.schema(ddl).options(header=True, escape='"', mode="FAILFAST", timestampFormat="yyyy-MM-dd'T'HH:mm:ssXXX").csv(f"Files/history/{name}.csv")
    create_once(f"history.{name}", csv, partitioned_by)

### Cell 9: Count the stores that received YG-2291

Reads through OneLake, once it shows both tables.

In [ ]:
import time

if spark.conf.get("spark.sql.catalog.readcat", None) is None:
    raise RuntimeError("Run cell 2 first. It sets up the Iceberg catalogs.")

deadline = time.monotonic() + 180
for table in ("history.lot_history", "history.shipment_history"):
    while not spark.catalog.tableExists(f"readcat.{table}"):
        if time.monotonic() > deadline:
            raise TimeoutError(f"After 3 minutes, OneLake still doesn't show {table}. Run this cell again.")
        time.sleep(10)

stores = spark.sql("""
    SELECT COUNT(DISTINCT store_id) AS stores
    FROM readcat.history.shipment_history
    WHERE lot_id = 'YG-2291'
""").first()["stores"]
print(f"{stores} stores received YG-2291")

### Cell 10: Look at the correction file

The carrier's late file adds 3 lines, moves 1 line to another store, and removes 20 duplicate lines.

In [ ]:
corrections = (
    spark.read.schema("action STRING, shipment_id STRING, line_no INT, shipped_at_utc TIMESTAMP, delivered_at_utc TIMESTAMP, origin_dc_id STRING, store_id STRING, carrier_id STRING, truck_id STRING, lot_id STRING, product_id STRING, supplier_id STRING, cases INT, supplier_cost_eur DECIMAL(10,2), status STRING, driver_note STRING")
    .options(header=True, escape='"', mode="FAILFAST", timestampFormat="yyyy-MM-dd'T'HH:mm:ssXXX")
    .csv("Files/history/corrections_2026-10-13.csv")
)
corrections.createOrReplaceTempView("corrections")

display(corrections.groupBy("action").count().orderBy("action"))
display(corrections.orderBy("action", "shipment_id", "line_no"))

### Cell 11: Insert the new lines

Skips any line that's already in the table, so a rerun inserts nothing.

In [ ]:
if spark.conf.get("spark.sql.catalog.writecat", None) is None:
    raise RuntimeError("Run cell 2 first. It sets up the Iceberg catalogs.")
if not spark.catalog.tableExists("corrections"):
    raise RuntimeError("Run cell 10 first. It loads the correction file.")

before = spark.table("writecat.history.shipment_history").count()
spark.sql("""
    INSERT INTO writecat.history.shipment_history
    SELECT c.shipment_id, c.line_no, c.shipped_at_utc, c.delivered_at_utc, c.origin_dc_id, c.store_id,
           c.carrier_id, c.truck_id, c.lot_id, c.product_id, c.supplier_id, c.cases,
           c.supplier_cost_eur, c.status, c.driver_note
    FROM corrections AS c
    WHERE c.action = 'insert'
      AND NOT EXISTS (
          SELECT 1
          FROM writecat.history.shipment_history AS h
          WHERE h.shipment_id = c.shipment_id AND h.line_no = c.line_no
      )
""")
after = spark.table("writecat.history.shipment_history").count()
print(f"{after - before} rows inserted")

display(spark.sql("SELECT * FROM writecat.history.shipment_history WHERE shipment_id = 'SHP-209481' ORDER BY line_no"))

### Cell 12: Update the moved line

The correction file moves SHP-207158 line 1 from STR-052 to STR-042.

In [ ]:
if spark.conf.get("spark.sql.catalog.writecat", None) is None:
    raise RuntimeError("Run cell 2 first. It sets up the Iceberg catalogs.")

line = "shipment_id = 'SHP-207158' AND line_no = 1"
query = f"SELECT store_id FROM writecat.history.shipment_history WHERE {line}"

before = spark.sql(query).first()["store_id"]
spark.sql(f"""
    UPDATE writecat.history.shipment_history
    SET store_id = 'STR-042'
    WHERE {line} AND store_id <> 'STR-042'
""")
after = spark.sql(query).first()["store_id"]
print(f"SHP-207158 line 1: {before} -> {after}")

### Cell 13: Delete the duplicate lines

Write the condition, then run the cell.

In [ ]:
if spark.conf.get("spark.sql.catalog.writecat", None) is None:
    raise RuntimeError("Run cell 2 first. It sets up the Iceberg catalogs.")
if not spark.catalog.tableExists("corrections"):
    raise RuntimeError("Run cell 10 first. It loads the correction file.")

# Replace None with a SQL condition, as a string, that matches the 20 delete rows
# in the corrections view on shipment_id and line_no.
condition = None

if condition is None:
    raise RuntimeError("Write the DELETE condition first. Exercise 2.3 has a hint.")

before = spark.table("writecat.history.shipment_history").count()
spark.sql(f"DELETE FROM writecat.history.shipment_history WHERE {condition}")
after = spark.table("writecat.history.shipment_history").count()
print(f"{before - after} rows deleted")

### Cell 14: Compare snapshots

Each write made a snapshot. Time travel counts the stores that received YG-2291 at the first snapshot and at the latest.

In [ ]:
if spark.conf.get("spark.sql.catalog.writecat", None) is None:
    raise RuntimeError("Run cell 2 first. It sets up the Iceberg catalogs.")

snapshots = spark.sql("""
    SELECT committed_at, snapshot_id, operation,
           summary['added-records'] AS added_records,
           summary['deleted-records'] AS deleted_records
    FROM writecat.history.shipment_history.snapshots
    ORDER BY committed_at
""")
display(snapshots)


def stores_at(snapshot_id):
    return spark.sql(f"""
        SELECT COUNT(DISTINCT store_id) AS stores
        FROM writecat.history.shipment_history VERSION AS OF {snapshot_id}
        WHERE lot_id = 'YG-2291'
    """).first()["stores"]


ids = [row["snapshot_id"] for row in snapshots.collect()]
print(f"{len(ids)} snapshots")
print(f"First snapshot: {stores_at(ids[0])} stores received YG-2291")
print(f"Latest snapshot: {stores_at(ids[-1])} stores received YG-2291")

### Cell 15: Check the corrections through OneLake

Waits until OneLake shows the latest snapshot, then checks that all three corrections are in.

In [ ]:
import time

if spark.conf.get("spark.sql.catalog.readcat", None) is None:
    raise RuntimeError("Run cell 2 first. It sets up the Iceberg catalogs.")

latest = spark.sql("""
    SELECT snapshot_id
    FROM writecat.history.shipment_history.snapshots
    ORDER BY committed_at DESC
    LIMIT 1
""").first()["snapshot_id"]

deadline = time.monotonic() + 180
while spark.sql(f"SELECT 1 FROM readcat.history.shipment_history.snapshots WHERE snapshot_id = {latest}").count() == 0:
    if time.monotonic() > deadline:
        raise TimeoutError(f"After 3 minutes, OneLake still doesn't show snapshot {latest} of history.shipment_history. Run this cell again.")
    time.sleep(10)

# The three changes in the correction file.
applied = spark.sql("""
    SELECT
        count_if(shipment_id = 'SHP-209481') AS inserted,
        count_if(shipment_id = 'SHP-207158' AND line_no = 1 AND store_id = 'STR-042') AS updated,
        count_if(shipment_id IN ('SHP-208999', 'SHP-209240')) AS duplicates
    FROM readcat.history.shipment_history
""").first()
if (applied["inserted"], applied["updated"], applied["duplicates"]) != (3, 1, 0):
    raise RuntimeError(
        f"The correction file isn't fully applied: {applied['inserted']} of 3 lines inserted, "
        f"{applied['updated']} of 1 line updated, {applied['duplicates']} duplicate lines left. "
        "Finish cells 11 to 13, then run this cell again."
    )

stores = spark.sql("""
    SELECT COUNT(DISTINCT store_id) AS stores
    FROM readcat.history.shipment_history
    WHERE lot_id = 'YG-2291'
""").first()["stores"]
print(f"{stores} stores received YG-2291, read through OneLake")

## Exercise 2.4: Trace related lots with DuckDB

### Cell 16: Install DuckDB

In [ ]:
%pip install -q "duckdb>=1.5.6,<2"

### Cell 17: Attach OneLake to DuckDB

DuckDB reads the same Iceberg tables through the REST catalog. Nothing is copied.

In [ ]:
import duckdb

ctx = notebookutils.runtime.context
warehouse = f"{ctx['defaultLakehouseWorkspaceId']}/{ctx['defaultLakehouseId']}"

con = duckdb.connect()
con.execute("INSTALL iceberg; LOAD iceberg; INSTALL azure; LOAD azure; INSTALL httpfs; LOAD httpfs;")
con.execute("SET TimeZone = 'UTC'")
con.execute(
    "CREATE OR REPLACE SECRET onelake_catalog (TYPE ICEBERG, TOKEN ?)",
    [notebookutils.credentials.getToken("storage")],
)
con.execute(f"""
    ATTACH '{warehouse}' AS onelake (
        TYPE ICEBERG,
        SECRET onelake_catalog,
        ENDPOINT 'https://onelake.table.fabric.microsoft.com/iceberg',
        ACCESS_DELEGATION_MODE 'vended_credentials',
        STAGE_CREATE_TABLES false,
        DISABLE_MULTI_TABLE_COMMIT true
    )
""")

tables = con.execute("SHOW ALL TABLES").df()
display(tables.loc[tables["database"] == "onelake", ["schema", "name"]])

### Cell 18: Query the batch window on the UTC clock

SUP-014 runs line AVD-L2 every Sunday from 22:00 to 02:00. The timestamps are in UTC.

In [ ]:
if "con" not in globals():
    raise RuntimeError("Run cells 16 and 17 first. They install DuckDB and attach OneLake.")

utc_clock = con.execute("""
    WITH lots AS (
        SELECT lot_id, production_start_utc AT TIME ZONE 'UTC' AS start_utc
        FROM onelake.history.lot_history
        WHERE supplier_id = 'SUP-014'
          AND production_line = 'AVD-L2'
          -- 90 days before 2026-10-13 06:00 Paris time
          AND production_start_utc >= TIMESTAMPTZ '2026-07-15 04:00:00+00'
    )
    SELECT lot_id, start_utc
    FROM lots
    -- dayofweek: 0 is Sunday, 1 is Monday
    WHERE (dayofweek(start_utc) = 0 AND hour(start_utc) >= 22)
       OR (dayofweek(start_utc) = 1 AND hour(start_utc) < 2)
    ORDER BY start_utc
""").df()

found = "YG-2291" in set(utc_clock["lot_id"])
print(f"{len(utc_clock)} lots on the UTC clock. YG-2291 is {'in the list' if found else 'missing'}.")
display(utc_clock)

### Cell 19: Query the batch window on the Paris clock

Contoso runs on Paris time. The cell saves the result as `related_lots`.

In [ ]:
if "con" not in globals():
    raise RuntimeError("Run cells 16 and 17 first. They install DuckDB and attach OneLake.")

con.execute("""
    CREATE OR REPLACE TEMP VIEW related_lots AS
    WITH lots AS (
        SELECT lot_id, production_start_utc AT TIME ZONE 'Europe/Paris' AS start_paris
        FROM onelake.history.lot_history
        WHERE supplier_id = 'SUP-014'
          AND production_line = 'AVD-L2'
          AND production_start_utc >= TIMESTAMPTZ '2026-07-15 04:00:00+00'
    )
    SELECT lot_id, start_paris
    FROM lots
    WHERE (dayofweek(start_paris) = 0 AND hour(start_paris) >= 22)
       OR (dayofweek(start_paris) = 1 AND hour(start_paris) < 2)
""")
related = con.execute("SELECT * FROM related_lots ORDER BY start_paris").df()

found = "YG-2291" in set(related["lot_id"])
print(f"{len(related)} related lots on the Paris clock. YG-2291 is {'in the list' if found else 'missing'}.")
display(related)

### Cell 20: Find the stores the related lots reached

In [ ]:
if "con" not in globals():
    raise RuntimeError("Run cells 16 and 17 first. They install DuckDB and attach OneLake.")
if not con.execute("SELECT count(*) FROM duckdb_views() WHERE view_name = 'related_lots'").fetchone()[0]:
    raise RuntimeError("Run cell 19 first. It saves the related lots.")

reach = con.execute("""
    SELECT l.region, COUNT(DISTINCT s.store_id) AS stores
    FROM onelake.history.shipment_history AS s
    JOIN related_lots AS r ON r.lot_id = s.lot_id
    JOIN onelake.dbo.locations AS l ON l.location_id = s.store_id
    GROUP BY l.region
    ORDER BY l.region
""").df()

print(f"The related lots reached {int(reach['stores'].sum())} stores in {len(reach)} regions")
display(reach)

## Exercise 2.5: Read shipment notes with AI Functions

### Cell 21: Apply the AI Functions settings

Runtime 2.0 needs these temporary settings once per session.

In [ ]:
import synapse.ml.spark.aifunc as aifunc

aifunc.default_conf.set_deployment_name("gpt-5.1")
aifunc.default_conf.set_reasoning_effort("low")
aifunc.default_conf.reset_temperature()

print(f"Deployment: {aifunc.default_conf.get_deployment_name()}")
print(f"Temperature: {aifunc.default_conf.get_temperature()}")

### Cell 22: Extract temperatures and durations

Reads the 30 driver notes on related-lot shipments. 13 of them are in French.

In [ ]:
if spark.conf.get("spark.sql.catalog.readcat", None) is None:
    raise RuntimeError("Run cell 2 first. It sets up the Iceberg catalogs.")
if "aifunc" not in globals():
    raise RuntimeError("Run cell 21 first. It applies the AI Functions settings.")

notes = spark.sql("""
    WITH lots AS (
        SELECT lot_id, qa_note, from_utc_timestamp(production_start_utc, 'Europe/Paris') AS start_paris
        FROM readcat.history.lot_history
        WHERE supplier_id = 'SUP-014'
          AND production_line = 'AVD-L2'
          AND production_start_utc >= TIMESTAMP '2026-07-15 04:00:00'
    ),
    related_lots AS (
        SELECT lot_id, qa_note
        FROM lots
        -- dayofweek: 1 is Sunday, 2 is Monday
        WHERE (dayofweek(start_paris) = 1 AND hour(start_paris) >= 22)
           OR (dayofweek(start_paris) = 2 AND hour(start_paris) < 2)
    )
    SELECT s.lot_id, s.store_id, s.status, s.driver_note
    FROM readcat.history.shipment_history AS s
    JOIN related_lots AS r ON r.lot_id = s.lot_id
    WHERE s.driver_note IS NOT NULL
""")

readings = notes.ai.extract(
    labels=[
        aifunc.ExtractLabel(
            label="temperature_c",
            description="The temperature in degrees Celsius. A comma can be the decimal separator.",
            max_items=1,
            type="number",
        ),
        aifunc.ExtractLabel(
            label="minutes",
            description="How many minutes the temperature alarm lasted. Leave it empty if the note gives no duration.",
            max_items=1,
            type="integer",
        ),
    ],
    input_col="driver_note",
    error_col="error",
).cache()

counts = readings.selectExpr(
    "count(*) AS notes",
    "count(temperature_c) AS temperatures",
    "count(minutes) AS durations",
).first()
print(f"{counts['notes']} notes: {counts['temperatures']} temperatures, {counts['durations']} durations")
display(readings.orderBy("lot_id", "store_id"))

### Cell 23: Summarize the QA notes

Turns each QA note on a related lot into one English sentence.

In [ ]:
if spark.conf.get("spark.sql.catalog.readcat", None) is None:
    raise RuntimeError("Run cell 2 first. It sets up the Iceberg catalogs.")
if "aifunc" not in globals():
    raise RuntimeError("Run cell 21 first. It applies the AI Functions settings.")

qa_notes = spark.sql("""
    WITH lots AS (
        SELECT lot_id, qa_note, from_utc_timestamp(production_start_utc, 'Europe/Paris') AS start_paris
        FROM readcat.history.lot_history
        WHERE supplier_id = 'SUP-014'
          AND production_line = 'AVD-L2'
          AND production_start_utc >= TIMESTAMP '2026-07-15 04:00:00'
    ),
    related_lots AS (
        SELECT lot_id, qa_note
        FROM lots
        -- dayofweek: 1 is Sunday, 2 is Monday
        WHERE (dayofweek(start_paris) = 1 AND hour(start_paris) >= 22)
           OR (dayofweek(start_paris) = 2 AND hour(start_paris) < 2)
    )
    SELECT lot_id, qa_note
    FROM related_lots
    WHERE qa_note IS NOT NULL
""")

summaries = qa_notes.ai.generate_response(
    prompt=(
        "Rewrite the QA note on lot {lot_id} as one plain English sentence. "
        "Translate it if it's in French. QA note: {qa_note}"
    ),
    is_prompt_template=True,
    output_col="summary",
    error_col="error",
)
display(summaries.orderBy("lot_id"))